In [ ]:
# Setup Environment & Compile C++ Engine
!pip install -q pybind11 pandas matplotlib

import os
# Clone the repository if it's not already in the Kaggle working directory
repo_dir = "/kaggle/working/zerocross-ai"
if not os.path.exists(repo_dir):
    !git clone https://github.com/muhammad-hassaan-aiml/zerocross-ai.git {repo_dir}

os.chdir(repo_dir)

# Compile the C++ engine safely on the Kaggle GPU
!mkdir -p build
os.chdir("build")
!cmake -Dpybind11_DIR=$(python3 -c "import pybind11; print(pybind11.get_cmake_dir())") ..
!cmake --build . -j 4
!cp zerocross_engine*.so ../python/
os.chdir("..")

In [ ]:
# Auto-Resume Logic for Multi-Session Runs
import os
import shutil

# Kaggle automatically mounts your datasets in /kaggle/input/
input_dir = "/kaggle/input/"
working_models_dir = "/kaggle/working/models"

os.makedirs(working_models_dir, exist_ok=True)

# Search for previous run files and copy them to the working directory
found_previous = False
if os.path.exists(input_dir):
    for root, dirs, files in os.walk(input_dir):
        if "best_model.pth" in files:
            print(f"Found previous session data in {root}!")
            print("Copying to working directory to resume training...")
            for file in files:
                if file.endswith(".pth") or file.endswith(".pt") or file.endswith(".csv"):
                    source_path = os.path.join(root, file)
                    shutil.copy(source_path, working_models_dir)
            found_previous = True
            break

if not found_previous:
    print("No previous dataset attached. Starting a fresh Session #1.")
else:
    print("Resume data loaded successfully!")

In [ ]:
# Run the 256-Channel AlphaZero Pipeline
!python python/pipeline.py \
    --iterations 25 \
    --concurrent-games 100 \
    --mcts-sims 400 \
    --eval-games 40 \
    --eval-sims 400 \
    --batch-size 512